In [100]:

import requests
import pandas as pd
import numpy as np

import json
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import os
import io

from understatapi import UnderstatClient

from thefuzz import process, fuzz

from scipy.stats import poisson

gw = 22

# 25/26


In [14]:
# !pip install selenium

In [16]:
players_raw = pd.read_csv('https://raw.githubusercontent.com/ilyandho/FPL-Optimal-Transfer/refs/heads/FantasyGo/FPL%20predictors/with%20new%20features/data/vaastav/data/2025-26/players_raw.csv')
players_1 = pd.read_csv('https://raw.githubusercontent.com/ilyandho/FPL-Optimal-Transfer/refs/heads/FantasyGo/FPL%20predictors/with%20new%20features/data/vaastav/data/2025-26/gws/gw1.csv')
v = pd.read_csv('https://github.com/ilyandho/FPL-Optimal-Transfer/raw/refs/heads/FantasyGo/FPL%20predictors/with%20new%20features/data/vaastav/data/2025-26/players/Aaron_Hickey_116/gw.csv')
master_fpl_understat_map = pd.read_csv('https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/main/Master.csv')

# player_gw_stats = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/player_gameweek_stats.csv')
# playerstats_1 = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/playerstats.csv')
players = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/players.csv')
teams = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/teams.csv')

git_data_olbauday_base = "https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/"

fpl_fixtures_url = 'https://fantasy.premierleague.com/api/fixtures/?event='

player_summary = 'https://fantasy.premierleague.com/api/element-summary/'

# bootstrap = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/').json()
# bootstrap

## Understat


In [17]:
# Initialize the client
with UnderstatClient() as understat:
    # Use .league() to specify the league, then .get_player_data() for the season
    data = understat.league(league="EPL").get_player_data(season="2025")

# This will return a list of dictionaries containing player stats
understat_data = pd.DataFrame(data)
understat_data.to_csv(f'./data/understat/understat_data_{gw}.csv')
understat_data

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup
0,8260,Erling Haaland,21,1846,20,20.03544656187296,4,3.155729355290532,83,12,0,0,F,Manchester City,18,17.75193990021944,21.18737766891718,2.850555630400777
1,13222,Thiago,21,1762,16,15.706132754683495,1,2.1366206761449575,50,11,4,0,F S,Brentford,11,11.139119647443295,13.84080659225583,2.6555524803698063
2,11363,Antoine Semenyo,20,1800,10,7.9325351398438215,3,2.4660277236253023,49,25,6,0,M,Bournemouth,9,6.410197442397475,10.571022145450115,3.088812258094549
3,5555,Dominic Calvert-Lewin,19,1314,9,8.426583101972938,1,1.31724401563406,39,13,1,0,F S,Leeds,7,6.904245397076011,9.248724179342389,1.3431714698672295
4,501,Danny Welbeck,20,1116,8,6.669347804039717,0,0.31755480915308,28,10,3,0,F S,Brighton,7,4.385841159150004,6.039988946169615,1.8120669340714812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,14219,Mohamadou Kanté,3,8,0,0.06551869213581085,0,0.10989412665367126,1,1,0,0,S,West Ham,0,0.06551869213581085,0.17541281878948212,0
485,14263,Joél Drakes-Thomas,2,2,0,0,0,0,0,0,0,0,S,Crystal Palace,0,0,0.3753929138183594,0.3753929138183594
486,14266,Bendito Mantato,1,14,0,0,0,0.03647809848189354,0,1,0,0,S,Manchester United,0,0,0.12379638105630875,0.12379638105630875
487,14281,Jaydon Banel,1,7,0,0,0,0,0,0,0,0,S,Burnley,0,0,0,0


In [18]:

# List of IDs from your data
player_ids = understat_data['id'].values #['8260', '13222', '11363'] # Haaland, Thiago, Semenyo, etc.
player_name_id = understat_data.set_index('id')['player_name']
player_name_id
all_history = []

with UnderstatClient() as understat:
    for p_id in player_ids:
        # This gets the season-by-season history for that specific ID
        history = understat.player(player=p_id).get_match_data()

        # # Add the player name back in so you know who is who
        for gw_ in history:
            gw_['understat_id'] = p_id
            gw_['understat_name'] = player_name_id[p_id]
            all_history.append(gw_)

# # Convert to a history DataFrame
history_df = pd.DataFrame(all_history)
history_df.to_csv(f'./data/understat/understat_hist_{gw}.csv', index=False)

## FPL


### GW 1


In [ ]:
all_player_gw_stats = []
for round_ in range(1,gw):
    player_gw_stats = pd.read_csv(f'{git_data_olbauday_base}GW{round_}/player_gameweek_stats.csv')
    player_gw_stats['round'] = round_
    all_player_gw_stats.append(player_gw_stats)

all_player_gw_stats = pd.concat(all_player_gw_stats, ignore_index=True)

all_teams = []
for round_ in range(1,gw):
    team_stats = pd.read_csv(f'{git_data_olbauday_base}GW{round_}/teams.csv')
    team_stats['round'] = round_
    all_teams.append(team_stats)

all_team_stats = pd.concat(all_teams, ignore_index=True)

all_players = []
for round_ in range(1,gw):
    team_stats = pd.read_csv(f'{git_data_olbauday_base}GW{round_}/players.csv')
    team_stats['round'] = round_
    all_players.append(team_stats)

players = pd.concat(all_players, ignore_index=True)

matches = []

for round_ in range(1,gw):
    data = requests.get(fpl_fixtures_url+str(round_)).json()

    matches = [*matches, *[{
                    'round': event['event'], 'team_id': event['id'], 'team_a': event['team_a'], 'team_h': event['team_h'],
                    'team_h_difficulty':event['team_h_difficulty'], 'team_a_difficulty':event['team_a_difficulty'], 'team_a_score': event['team_a_score'],
                    'team_h_score': event['team_h_score'], 'kickoff_time': event['kickoff_time']
                     } for event in data]]
    #     []
    # matches.append()
match_details = pd.DataFrame(matches)
match_details.to_csv(f'./data/match_details_{gw}.csv', index=False)

In [20]:

player_gw_stats_clean = all_player_gw_stats[all_player_gw_stats['status'].isin(['a', 'd'])]  # Remove unavailable players
player_gw_stats_clean.columns.tolist()
gw_stats_cols =  [
                    'id', 'first_name', 'second_name', 'web_name', 'now_cost',  'selected_by_percent',  'form',  'event_points', 'transfers_in_event',
                    'transfers_out_event', 'value_form', 'ep_next', 'ep_this', 'chance_of_playing_next_round', 'chance_of_playing_this_round',  'gw',
                    'total_points', 'minutes',  'goals_scored',  'assists',  'clean_sheets',  'goals_conceded', 'yellow_cards',  'red_cards',  'saves',
                    'starts',  'bonus',  'bps',  'transfers_in',  'transfers_out', 'expected_goals',  'expected_assists',  'expected_goal_involvements',
                    'expected_goals_conceded',  'influence',  'creativity',  'threat',  'ict_index',  'tackles',  'clearances_blocks_interceptions',
                    'recoveries',  'defensive_contribution',  'round',
                ]

player_gw_stats_clean = player_gw_stats_clean[gw_stats_cols]
player_gw_stats_clean[['chance_of_playing_this_round', 'chance_of_playing_next_round']]= player_gw_stats_clean[['chance_of_playing_this_round', 'chance_of_playing_next_round']].fillna(100)
ids_with_details = players['player_id'].unique().tolist()
player_gw_stats_clean = player_gw_stats_clean[player_gw_stats_clean['id'].isin(ids_with_details)]
player_gw_stats_clean = player_gw_stats_clean.merge(players[['player_id','team_code', 'position', 'round']], left_on=['id', 'round'], right_on=['player_id', 'round'], how='left')
player_gw_stats_clean = player_gw_stats_clean.dropna(subset=['team_code'])
team_data = all_team_stats.rename({'code': 'team_code', 'id': 'team_id', 'name':'team', 'short_name':'team_short_name', 'ep_this': 'xP', 'ep_next':'xP_next'}, axis=1)
player_gw_stats_clean = player_gw_stats_clean.merge(team_data[[
  'team_code', 'team_id', 'team', 'round', 'team_short_name', 'elo', 'strength', 'strength_overall_home', 'strength_overall_away', 'strength_attack_home', 'strength_attack_away',
    'strength_defence_home', 'strength_defence_away']], on=['round','team_code'], how='left')

# Reshape matches so every team_id has its own row per match
match_details = match_details.rename(columns={'team_id': 'match_id'})
matches_melted = match_details.melt(
    id_vars=['match_id', 'round', 'team_h_difficulty', 'team_a_difficulty', 'team_a_score', 'team_h_score', 'kickoff_time'],
    value_vars=['team_a', 'team_h'],
    var_name='side',
    value_name='team_id'
)

player_match_df = player_gw_stats_clean.merge(matches_melted, on=['team_id', 'round'], how='left')
player_match_df

,id,first_name,second_name,web_name,now_cost,selected_by_percent,form,event_points,transfers_in_event,transfers_out_event,...,strength_attack_away,strength_defence_home,strength_defence_away,match_id,team_h_difficulty,team_a_difficulty,team_a_score,team_h_score,kickoff_time,side
0,1,David,Raya Martín,Raya,6.0,36.9,3.2,10,418466,56691,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
1,2,Kepa,Arrizabalaga Revuelta,Arrizabalaga,4.1,0.4,0.0,0,793,2256,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
2,4,Tommy,Setford,Setford,3.9,0.2,0.0,0,2397,1906,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
3,6,William,Saliba,Saliba,6.0,10.3,0.5,9,25445,160545,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
4,8,Jurriën,Timber,J.Timber,6.5,33.0,2.8,0,365426,536834,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11444,547,Enzo,Le Fée,E.Le Fée,4.9,1.0,2.5,0,3640,5038,...,1120,1100,1170,203,2,3,0,3,2026-01-07T19:30:00Z,team_a
11445,156,Charlie,Tasker,Tasker,3.8,0.1,0.0,0,266,189,...,1110,1210,1210,208,3,4,1,1,2026-01-07T19:30:00Z,team_a
11446,549,Dan,Neil,Neil,4.8,0.0,0.2,1,27,42,...,1120,1100,1170,203,2,3,0,3,2026-01-07T19:30:00Z,team_a
11447,551,Chris,Rigg,Rigg,4.7,0.0,1.2,1,79,115,...,1120,1100,1170,203,2,3,0,3,2026-01-07T19:30:00Z,team_a


In [21]:
unique_players = player_gw_stats_clean['id'].unique()

player_api_data = []

for pid in unique_players:
    # Fetch from API
    response = requests.get(f'{player_summary}{pid}/')
    data = response.json()  # assuming JSON response
    history_data = data['history']

    player_api_data = [*player_api_data, *history_data]

pd.DataFrame(player_api_data).rename({'element': 'id'}, axis=1).to_csv(f'./data/player_summary_{gw}.csv', index=False)

In [22]:
player_api_details = pd.read_csv(f'./data/player_summary_{gw}.csv')
understat_data = pd.read_csv(f'./data/understat/understat_hist_{gw}.csv')
understat_data = understat_data[understat_data['season'] == 2025]

# ids availabe from the api
live_ids = player_api_details['id'].unique().tolist()
player_match_df_ = player_match_df[player_match_df['id'].isin(live_ids)].copy()

fpl_understat_map = master_fpl_understat_map[(
                                                ~master_fpl_understat_map['25-26'].isna())
                                                & ~(master_fpl_understat_map['understat'].isna())].rename({'25-26': 'fpl_id', 'understat':'understat_id'}, axis=1)

player_match_df_['date'] = pd.to_datetime(player_match_df_['kickoff_time']).dt.date
understat_data['date'] = pd.to_datetime(understat_data['date']).dt.date
player_gw_stats_full = player_match_df_.merge(player_api_details[['id', 'round', 'selected', 'was_home', 'value', 'opponent_team']],
                                            on=['round', 'id'],
                                            how='left')

player_gw_stats_full = player_gw_stats_full.rename(columns={'player_id':'fpl_id'}).drop('id', axis=1)

player_gw_stats_full = player_gw_stats_full[player_gw_stats_full['fpl_id'].isin(fpl_understat_map['fpl_id'].unique().tolist())]

understat_data = understat_data.merge(fpl_understat_map[['fpl_id', 'understat_id']], on='understat_id', how='left').dropna()
player_gw_stats_full = player_gw_stats_full.merge(fpl_understat_map[['fpl_id', 'understat_id']],
                                                  on='fpl_id',
                                                  how='left')

player_gw_stats_full['full_name'] = (player_gw_stats_full['first_name'] + ' ' + player_gw_stats_full['second_name']).str.strip()

player_data = player_gw_stats_full.merge(understat_data[[
                                'goals', 'shots', 'xG','h_team', 'a_team',
                                'h_goals', 'a_goals', 'date', 'season', 'roster_id', 'xA',
                                'key_passes', 'npg', 'npxG', 'xGChain', 'xGBuildup',
                                'understat_id', 'understat_name', 'fpl_id']],
                                on=['date', 'fpl_id', 'understat_id'],
                                how='left'
                                )

player_data = player_data.rename(columns={'ep_this': 'xP', 'ep_next': 'xP_next'})

player_data.to_csv('./data/player_data.csv', index=False)

### Add Odds


In [23]:
player_data = pd.read_csv('./data/player_data.csv', low_memory=False)
player_data_clean = player_data.dropna()
# Load the current season's data directly from the source
season = "2526" # Change this for historical seasons
url = f"https://www.football-data.co.uk/mmz4281/{season}/E0.csv"

# It's good practice to use a custom User-Agent to avoid blocks
odds_data = pd.read_csv(url, low_memory=False)

# 1. Clean up dates in betting data
odds_data['Date'] = pd.to_datetime(odds_data['Date'], dayfirst=True).dt.date

# 2. Extract key columns (B365H = Bet365 Home Odds, B365D = Draw, B365A = Away)
odds_subset = odds_data[['Date', 'HomeTeam', 'AwayTeam', 'B365H', 'B365D', 'B365A']].copy()
odds_subset = odds_subset.rename(columns={'Date': 'date'})

def add_odds(row):
    win = round(1/row['B365H'], 5)
    draw = round(1/row['B365D'], 5)
    lose = round(1/row['B365A'], 5)

    # Normalize the probabilities (to make the probabilities sum to 100%)
    sum_percent = win + draw + lose
    win_prob = round(win/sum_percent, 3)
    draw_prob = round(draw/sum_percent, 3)
    lose_prob = round(lose/sum_percent, 3)

    return pd.Series([win_prob, draw_prob, lose_prob])

odds_subset[['win_prob', 'draw_prob', 'lose_prob']] = odds_subset.apply(add_odds, axis=1)

# Make sure the names match in the player_data df and odds_subset df
team_map = {
    'Bournemouth': 'Bournemouth',
    'Newcastle': 'Newcastle',
    'Fulham': 'Fulham',
    'West Ham': 'West Ham',
    'Burnley': 'Burnley',
    'Man City': 'Man City',
    'Crystal Palace': 'Crystal Palace',
    'Brentford': 'Brentford',
    'Arsenal': 'Arsenal',
    'Everton': 'Everton',
    'Chelsea': 'Chelsea',
    'Tottenham': 'Spurs',
    'Wolves': 'Wolves',
    'Aston Villa': 'Aston Villa',
    'Sunderland': 'Sunderland',
    'Leeds': 'Leeds',
    "Nott'm Forest": "Nott'm Forest",
    'Brighton': 'Brighton',
    'Man United': 'Man Utd',
    'Liverpool': 'Liverpool'
}

odds_subset['HomeTeam'] = odds_subset['HomeTeam'].map(team_map)
odds_subset['AwayTeam'] = odds_subset['AwayTeam'].map(team_map)

odds_melted = odds_subset.melt(
    id_vars=['date','B365H', 'B365D', 'B365A', 'win_prob', 'draw_prob', 'lose_prob'],
    value_vars=['HomeTeam', 'AwayTeam'],
    var_name='side',
    value_name='team'
)

odds_melted['date'] = odds_melted['date'].astype(str)

player_data_odds = player_data_clean.merge(odds_melted[['date', 'win_prob', 'draw_prob', 'lose_prob','team']],
                  on=['date', 'team'],
                  how='left'
                  )

# 1. Sort by player and round to ensure the sequence is correct
player_data_odds = player_data_odds.sort_values(['fpl_id', 'round'])

# 2. Group by player and apply the difference
player_data_odds['ownership_change'] = player_data_odds.groupby('fpl_id')['selected'].diff().fillna(0)
player_data_odds['pts_bonus'] = player_data_odds['total_points'] - player_data_odds['bonus']

def ownership_change(row):
    net_transfers = row['transfers_in'] - row['transfers_out']
    total_transfers = row['transfers_in'] + row['transfers_out']
    net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

    return net_transfers_pct

player_data_odds['percenatge_net_transfers'] = player_data_odds.apply(ownership_change, axis=1)


player_data_odds.to_csv('./data/final_player_data.csv')

## Current game week


### Get Odds from WilliamHill


In [24]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# Setup Driver (Adding options for stability)
options = webdriver.EdgeOptions()
# options.add_argument("--headless") # Uncomment to run without a window
driver = webdriver.Edge(options=options)

premLeague = "https://sports.williamhill.com/betting/en-gb/football/competitions/OB_TY295/English-Premier-League/matches/OB_MGMB/Match-Betting"
driver.get(premLeague)

# Explicit Wait: Wait up to 10 seconds for the match rows to appear
wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.sp-o-market--default")))

matches = driver.find_elements(By.CSS_SELECTOR, "article.sp-o-market--default")
# details = pd.DataFrame(columns=['team_h', 'team_h', 'win_prob', 'draw_prob', 'lose_prob'])
odds_list = []
for match in matches:
    try:
        # 1. Extract teams
        teams_text = match.find_element(By.CSS_SELECTOR, 'main.sp-o-market__title span').text
        if ' v ' not in teams_text: continue

        h_team_name = teams_text.split(' v ')[0]
        a_team_name = teams_text.split(' v ')[1]

        # 2. Extract odds
        odds_els = match.find_elements(By.CSS_SELECTOR, 'section.sp-o-market__buttons .sp-betbutton > span')

        if len(odds_els) < 3: continue

        decimal_odds = []
        for btn in odds_els:
            text = btn.text.strip()
            if text == 'EVS' or text == '1/1':
                decimal_odds.append(2.0)
            elif '/' in text:
                n, d = map(int, text.split('/'))
                decimal_odds.append((n / d) + 1)
            else:
                decimal_odds.append(float(text))

        # 3. Probability Calculation
        h_raw = 1 / decimal_odds[0]
        d_raw = 1 / decimal_odds[1]
        a_raw = 1 / decimal_odds[2]

        margin_total = h_raw + d_raw + a_raw

        # print('-----------------------------------------------------')
        # 4. Append to DataFrame
        match_row = {
            'team_h_name': h_team_name,
            'team_a_name': a_team_name,
            'win_prob': round(h_raw / margin_total, 3),
            'draw_prob':round(d_raw / margin_total, 3),
            'lose_prob': round(a_raw / margin_total, 3)
        }
        odds_list.append(match_row)
        # print(match_row)

    except Exception as e:
        print(f"Skipping a match due to error: {e}")


odds_details = pd.DataFrame(odds_list)

driver.quit()

odds_details.to_csv(f'./data/odds_{gw}.csv', index=False)

odds_details

,team_h_name,team_a_name,win_prob,draw_prob,lose_prob
0,Man Utd,Man City,0.243,0.243,0.514
1,Chelsea,Brentford,0.559,0.239,0.202
2,Leeds,Fulham,0.412,0.299,0.289
3,Liverpool,Burnley,0.761,0.155,0.085
4,Sunderland,Crystal Palace,0.341,0.307,0.351
5,Tottenham,West Ham,0.550,0.246,0.203
6,Nottingham Forest,Arsenal,0.168,0.237,0.596
7,Wolves,Newcastle,0.213,0.246,0.541
8,Aston Villa,Everton,0.548,0.259,0.194
9,Brighton,Bournemouth,0.498,0.246,0.256


### FPL Live Data


In [25]:

# 'https://fantasy.premierleague.com/api/element-summary/21'
    # 'round', 'web_name', 'position',


# 'https://fantasy.premierleague.com/api/fixtures/?event=21'
    # 'team_h_difficulty', 'team_a_difficulty', 'opponent_team', 'was_home',

# https://fantasy.premierleague.com/api/bootstrap-static/
    # elements
        # 'fpl_id',
        # chance_of_playing_next_round', 'chance_of_playing_this_round'
        # 'xP_next', 'xP',
        # 'form', 'value_form', 'value',  ',
        # 'selected_by_percent', 'transfers_in', 'transfers_out',  'selected', 'ownership_change', 'percenatge_net_transfers',
        # 'influence', 'creativity', 'threat', 'ict_index',
    # teams
        # 'strength','strength_overall_home', 'strength_overall_away', 'strength_attack_home', 'strength_attack_away',
        # 'strength_defence_home', 'strength_defence_away',  'team_id', 'team',

# http://api.clubelo.com/2026-01-06
    # 'elo'

# WilliamHill
    # 'win_prob', 'draw_prob', 'lose_prob',

In [26]:
current_player_summary = requests.get(f'https://fantasy.premierleague.com/api/bootstrap-static/').json()
events = pd.DataFrame(requests.get(f'https://fantasy.premierleague.com/api/fixtures/?event={gw}').json())
elo_df = pd.read_csv('http://api.clubelo.com/2026-01-11')

In [27]:
teams_data = pd.DataFrame(current_player_summary['teams'])
players_data = pd.DataFrame(current_player_summary['elements'])
total_players = current_player_summary['total_players']
players_data [['chance_of_playing_this_round', 'chance_of_playing_next_round']]= players_data[['chance_of_playing_this_round', 'chance_of_playing_next_round']].fillna(100)
players_data = players_data[(players_data['status'].isin(['a', 'd']))]  # Remove unavailable players
players_data['selected'] = (players_data['selected_by_percent'].astype(float)/100 * total_players).round().astype(int)

teams_ids = teams[['id', 'name']].rename({'id': 'team', 'name': 'team_name'}, axis=1)
teams_data = teams_data.rename({'id': 'team', 'name': 'team_name'}, axis=1)
players_data_ = players_data.merge(teams_data[[
                       'team', 'team_name', 'strength','strength_overall_home', 'strength_overall_away', 'strength_attack_home', 'strength_attack_away',
                        'strength_defence_home', 'strength_defence_away']],
                        on='team',
                        how='left'
                        )
players_data_ = players_data_.rename({'element_type': 'position',  'ep_next': 'xP_next', 'now_cost': 'value',}, axis=1)
# Get event details
events_ = events[['event', 'team_a', 'team_h', 'team_h_difficulty', 'team_a_difficulty']]

def add_team_details(row):
    team = row['team']
    event_ = events_[(events_['team_a'] == team) | (events_['team_h'] == team)]

    was_home = team == event_['team_h'].values[0]
    opponent_team = event_['team_a'].values[0] if was_home else event_['team_h'].values[0]
    event = event_['event'].values[0]

    return pd.Series([
                        event_['team_h'].values[0],
                        event_['team_a'].values[0],
                        event_['team_h_difficulty'].values[0],
                        event_['team_a_difficulty'].values[0],
                        was_home,
                        opponent_team,
                        event])

players_data_[['team_h', 'team_a', 'team_h_difficulty', 'team_a_difficulty', 'was_home', 'opponent_team', 'round']] = players_data_.apply(add_team_details, axis=1)

# Add Elo
pl_elo = elo_df[(elo_df['Country'] == 'ENG') & (elo_df['Level'] == 1)].sort_values('Elo', ascending=False)

fpl_teams = {
    'Arsenal' : 'Arsenal',
    'Man City' : 'Man City',
    'Liverpool' :'Liverpool',
    'Aston Villa' : 'Aston Villa' ,
    'Chelsea' : 'Chelsea',
    'Newcastle' : 'Newcastle',
    'Brighton' : 'Brighton',
    'Man United' : 'Man Utd',
    'Man Utd' : 'Man Utd',
    'Brentford' : 'Brentford',
    'Tottenham' : 'Spurs',
    'Everton' : 'Everton',
    'Crystal Palace' : 'Crystal Palace',
    'Fulham' : 'Fulham',
    'Bournemouth' : 'Bournemouth',
    'Forest' : "Nott'm Forest",
    'Nottingham Forest': "Nott'm Forest",
    'Leeds' : 'Leeds',
    'West Ham' : 'West Ham',
    'Burnley' : 'Burnley',
    'Sunderland' : 'Sunderland',
    'Wolves' : 'Wolves'
}

pl_elo['team_name'] = pl_elo['Club'].map(fpl_teams)
pl_elo = pl_elo.rename({'Elo': 'elo'}, axis=1)

players_data_ = players_data_.merge(pl_elo[['team_name', 'elo']],
                    on='team_name',
                    how='left')

# Adds odds details
odds_details = pd.read_csv('./odds_21.csv', index_col=0)
odds_details = odds_details.rename({'a_team': 'team_a_name', 'h_team': 'team_h_name'}, axis=1)

# Make sure the team names match
odds_details['team_a_name']  = odds_details['team_a_name'].map(fpl_teams)
odds_details['team_h_name']  = odds_details['team_h_name'].map(fpl_teams)

id_team_map = pd.Series(teams['name'].values, index=teams['id']).to_dict()

players_data_['team_h_name'] =  players_data_['team_h'].map(id_team_map)
players_data_['team_a_name'] =  players_data_['team_a'].map(id_team_map)

players_data_ = players_data_.merge(odds_details,
                    on=['team_h_name', 'team_a_name'],
                    how='left')


players_data_ = players_data_.rename({'id': 'fpl_id', 'team': 'team_id', 'team_name': 'team'}, axis=1)


# 1. Sort by player and round to ensure the sequence is correct
players_data_ = players_data_.sort_values(['fpl_id', 'round'])

def ownership_change(row):
    net_transfers = row['transfers_in'] - row['transfers_out']
    total_transfers = row['transfers_in'] + row['transfers_out']
    net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

    return net_transfers_pct

players_data_['percenatge_net_transfers'] = players_data_.apply(ownership_change, axis=1)

players_data_.to_csv(f'./data/player_data{gw}.csv')

### Add to main DF


In [28]:
target_cols = [
    'first_name', 'second_name',
    'fpl_id', 'round', 'web_name', 'position', 'team_id', 'team', 'opponent_team', 'was_home', 'xP_next',
    'selected_by_percent','value', 'form', 'value_form', 'chance_of_playing_next_round', 'chance_of_playing_this_round',
    'transfers_in', 'transfers_out',  'selected', 'percenatge_net_transfers', #'ownership_change',
    'influence', 'creativity', 'threat', 'ict_index', 'elo', 'strength',
    'strength_overall_home', 'strength_overall_away', 'strength_attack_home', 'strength_attack_away', 'strength_defence_home', 'strength_defence_away',
    'team_h_difficulty', 'team_a_difficulty', 'win_prob', 'draw_prob', 'lose_prob'
]
old_data = pd.read_csv('./data/final_player_data.csv', index_col=0)
new_data = pd.read_csv(f'./data/player_data{gw}.csv', index_col=0)

new_filtered = new_data[target_cols]

# 3. Append to your old data
# Use ignore_index=True to create a fresh index for the combined dataset
combined_df = pd.concat([old_data, new_filtered], ignore_index=True)
combined_df.to_csv('./data/combined_rounds.csv', index=False)
combined_df

,first_name,second_name,web_name,now_cost,selected_by_percent,form,event_points,transfers_in_event,transfers_out_event,value_form,...,npxG,xGChain,xGBuildup,understat_name,win_prob,draw_prob,lose_prob,ownership_change,pts_bonus,percenatge_net_transfers
0,David,Raya Martín,Raya,6.0,36.9,3.2,10.0,418466.0,56691.0,0.5,...,0.0,0.086901,0.086901,David Raya,0.260,0.260,0.480,0.0,7.0,0.463731
1,David,Raya Martín,Raya,5.5,21.5,8.0,6.0,77335.0,61830.0,1.5,...,0.0,0.204982,0.204982,David Raya,0.748,0.152,0.100,752723.0,6.0,0.492786
2,David,Raya Martín,Raya,5.5,21.8,6.0,2.0,38830.0,17474.0,1.1,...,0.0,0.108124,0.058624,David Raya,0.432,0.271,0.297,122330.0,2.0,-0.028978
3,David,Raya Martín,Raya,5.5,23.5,6.0,6.0,33143.0,28342.0,1.1,...,0.0,0.895559,0.895559,David Raya,0.690,0.191,0.119,358795.0,6.0,0.375592
4,David,Raya Martín,Raya,5.5,24.0,4.0,2.0,95889.0,32409.0,0.7,...,0.0,0.000000,0.000000,David Raya,0.510,0.242,0.248,-3127.0,2.0,0.179227
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6749,Valentín,Castellanos,Taty,NaN,0.0,2.0,NaN,NaN,NaN,0.4,...,NaN,NaN,NaN,NaN,0.555,0.244,0.201,NaN,NaN,0.753492
6750,Alysson Edward Franco,da Rocha dos Santos,Alysson,NaN,0.0,0.0,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,0.518,0.267,0.215,NaN,NaN,0.838710
6751,George,King,King,NaN,0.0,0.0,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,0.328,0.306,0.367,NaN,NaN,0.871795
6752,Max,Alleyne,Alleyne,NaN,0.0,4.0,NaN,NaN,NaN,0.9,...,NaN,NaN,NaN,NaN,0.236,0.236,0.527,NaN,NaN,0.789030


### Get CBITR & PER_90


In [ ]:
combined_data = pd.read_csv('./data/combined_rounds.csv')

pos_map = {
    'Goalkeeper': 'GK',
    '1': 'GK',
    'Defender': 'DEF',
    '2': 'DEF',
    'Midfielder': 'MID',
    '3': 'MID',
    'Forward': 'FWD',
    '4': 'FWD'
}
combined_data['position'] = combined_data['position'].map(pos_map)

# Get the total CBIT actions for the season so far
## Cater for the mssing rounds before rolling
# 1. Get all unique players and all unique rounds
all_players = combined_data['fpl_id'].unique()
all_rounds = range(combined_data['round'].min(), combined_data['round'].max() + 1)

# 2. Create a MultiIndex of every player x every round
multi_idx = pd.MultiIndex.from_product([all_players, all_rounds], names=['fpl_id', 'round'])

# 3. Reindex the dataframe
# This inserts "empty" rows for missing player/round combinations
combined_data = combined_data.set_index(['fpl_id', 'round']).reindex(multi_idx).reset_index()

# Fill statistical columns with 0
# stat_columns = ['goals_scored', 'goals_conceded', 'expected_goals_conceded']
combined_data = combined_data.fillna(0)

# Sort to ensure chronological order for the rolling window
combined_data = combined_data.sort_values(['fpl_id', 'round'])

# Get the sum of CBIT actions over the all the previous rounds
combined_data.sort_values(['fpl_id', 'round'], inplace=True)
# 2. Group by player and apply cumsum to relevant metrics
metrics_to_cumulate = ['recoveries', 'minutes',  'tackles', 'clearances_blocks_interceptions']

for metric in metrics_to_cumulate:
    col_name = f'season_{metric}'
    combined_data[col_name] = combined_data.groupby('fpl_id')[metric].transform(lambda x: x.shift(1).cumsum())

numeric_cols = ['season_recoveries', 'season_minutes',  'season_tackles', 'season_clearances_blocks_interceptions']
for col in numeric_cols:
    combined_data[col] = pd.to_numeric(combined_data[col], errors='coerce').fillna(0)

# Base CBIT actions (Clearances + Blocks + Interceptions + Tackles)
combined_data['cbit_actions'] = combined_data['season_tackles'] + combined_data['season_clearances_blocks_interceptions']

# Total Defensive Contribution (for Defenders and Midfielders includes Recoveries)
combined_data['total_cbitr_actions'] = combined_data.apply(
            lambda x: x['cbit_actions'] + x['season_recoveries'] if x['position'] in ['DEF', 'MID'] else x['cbit_actions'],
            axis=1
        )



# Calculate volume per 90 for the probability model
combined_data['cbitr_volume_90'] = (combined_data['total_cbitr_actions'] / combined_data['season_minutes'].replace(0, np.nan)) * 90

def estimate_cbit_prob(row):
    """
    Uses the cumulative rate (historical feature) to predict the
    probability of hitting the threshold in the next match.
    """
    threshold = 10 if row['position'] == 'DEF' else 12

    # Require a sample of at least 3 games (270 mins) for the feature to be considered reliable
    if row['season_minutes'] < 270:
        return 0.0

    # lam is the 'historical feature' representing expected actions in 90 mins
    lam = row['cbitr_volume_90']

    # Poisson survival function: P(X >= threshold)
    return round(poisson.sf(threshold - 1, lam), 4)

combined_data['cbitr_probability'] = combined_data.apply(estimate_cbit_prob, axis=1)

# combined_data[['web_name', 'position', 'tackles', 'clearances_blocks_interceptions', 'recoveries', 'season_recoveries', 'season_minutes',  'season_tackles', 'season_clearances_blocks_interceptions','cbit_actions', 'total_cbitr_actions', 'cbitr_volume_90', 'cbitr_probability']]
combined_data.to_csv('./data/combined_rounds.csv')

## Roll values


In [215]:
# player_data_odds = pd.read_csv('./data/final_player_data.csv')
combined_data = pd.read_csv('./data/combined_rounds.csv', index_col=0)
combined_data['opponent_team_name'] = combined_data.apply(lambda x: x['h_team'] if x['was_home'] else x['a_team'], axis=1)
rolling_features = [
    'creativity', 'influence', 'threat', 'minutes',  'pts_bonus', 'total_points',
    'expected_goals', 'expected_assists', 'xP',
    'expected_goals_conceded', 'goals_conceded', 'goals_scored',
    'shots', 'key_passes', 'npg', 'npxG','goals', 'shots', 'xG','xA',
    'saves', 'starts', 'yellow_cards', 'red_cards','assists', 'clean_sheets',

    'value', 'ict_index', 'selected', 'transfers_in', 'transfers_out',
    'ownership_change', 'percenatge_net_transfers',
    'xGChain', 'xGBuildup', 'expected_goal_involvements', 'form',
    'clearances_blocks_interceptions', 'tackles', 'recoveries','selected_by_percent',
    'goals_conceded','goals_scored',

    'strength', 'strength_overall_home', 'strength_overall_away',
    'strength_attack_home', 'strength_attack_away', 'strength_defence_home', 'strength_defence_away',
]

player_df = combined_data.copy()

# Fill statistical columns with 0
# stat_columns = ['goals_scored', 'goals_conceded', 'expected_goals_conceded']
player_df_complete = player_df.fillna(0)

# Sort to ensure chronological order for the rolling window
player_df_complete = player_df_complete.sort_values(['fpl_id', 'round']).reset_index(drop=True)

# Averagae values
new_features = {}
for col in rolling_features:
    # The rolling window now spans actual calendar rounds
    grp = player_df_complete.groupby('fpl_id')[col]

    new_features[f'{col}_1'] = grp.transform(
        lambda x: x.shift(1).rolling(1, min_periods=1).sum()
    )

    new_features[f'{col}_3'] = grp.transform(
        lambda x: (x.shift(1).rolling(3, min_periods=1).sum())/3
    )

    new_features[f'{col}_5'] = grp.transform(
        lambda x: (x.shift(1).rolling(5, min_periods=1).sum())/5
    )

# Join all new columns at once (no fragmentation)
player_df_complete = pd.concat(
    [player_df_complete, pd.DataFrame(new_features)],
    axis=1
)


### Weighted Averages


In [ ]:

# Estimate the chances of defensive contributions
# Base CBIT actions (Clearances + Blocks + Interceptions + Tackles)
# Base CBIT actions
player_df_complete['cbit'] = (
    player_df_complete['tackles'].fillna(0) +
    player_df_complete['clearances_blocks_interceptions'].fillna(0)
)

# Initialise CBITR with CBIT
player_df_complete['cbitr'] = player_df_complete['cbit']

# Add recoveries for MID and FWD only
mask_add_recoveries = player_df_complete['position'].isin(['MID', 'FWD'])
player_df_complete.loc[mask_add_recoveries, 'cbitr'] += (
    player_df_complete.loc[mask_add_recoveries, 'recoveries'].fillna(0)
)

# Zero out GK explicitly
player_df_complete.loc[player_df_complete['position'] == 'GK', 'cbitr'] = 0

# ----  minutes-adjusted EWMA per 90 ----
player_df_complete['cbitr_ewm_90'] = (
    player_df_complete
    .groupby('fpl_id')
    .apply(
        lambda g: (
            (g['cbitr'].shift(1) * g['minutes'].shift(1) / 90)
            .ewm(span=6, adjust=False)
            .mean()
        )
    )
    .reset_index(level=0, drop=True)
)


# Initialize
player_df_complete['cbitr_ewm_probability'] = 0.0

# Mask for eligible players (exclude GK)
mask = player_df_complete['position'] != 'GK'

# Thresholds
thresholds = np.where(player_df_complete['position'] == 'DEF', 10, 12)

# Mask eligible players with enough minutes
eligible = mask & (player_df_complete['season_minutes'] >= 270)

# Compute probabilities
player_df_complete.loc[eligible, 'cbitr_ewm_probability'] = poisson.sf(
    thresholds[eligible] - 1,
    player_df_complete.loc[eligible, 'cbitr_ewm_90']
).round(4)

## GET: ewm averages for features
EWMA_SPANS = {
    # Involvement
    'minutes': 3, 'starts': 3, 'shots': 3, 'key_passes': 3, 'saves': 3,

    # Attacking quality
    'expected_goals': 5, 'expected_assists': 5, 'xG': 5, 'xA': 5,
    'npxG': 5, 'npg': 5, 'xGChain': 5, 'xGBuildup': 5,
    'expected_goal_involvements': 5, 'xP': 5,
    'creativity': 5, 'threat': 5, 'influence': 5, 'ict_index': 5, 'form': 5,

    # Defensive
    'clearances_blocks_interceptions': 6, 'tackles': 6, 'recoveries': 6,
    'expected_goals_conceded': 6, 'goals_conceded': 6, 'clean_sheets': 6,

    # Discrete
    'goals': 8, 'assists': 8, 'yellow_cards': 8,
    'red_cards': 10, 'pts_bonus': 6, 'total_points': 6,

    # Market
    'selected': 10, 'selected_by_percent': 10,
    'transfers_in': 10, 'transfers_out': 10,
    'ownership_change': 10, 'percenatge_net_transfers': 10,
    'value': 10,

    # Team strength
    'strength': 10,
    'strength_overall_home': 15, 'strength_overall_away': 15,
    'strength_attack_home': 15, 'strength_attack_away': 15,
    'strength_defence_home': 15, 'strength_defence_away': 15,
}

for col, span in EWMA_SPANS.items():
    player_df_complete[f'{col}_ewm'] = (
        player_df_complete
        .groupby('fpl_id')[col]
        .transform(lambda x: x.shift(1).ewm(span=span, adjust=False).mean())
    )


In [234]:
player_df_complete = player_df_complete.copy()
# Ownership change per player
player_df_complete['ownership_change'] = (
    player_df_complete
    .groupby('fpl_id')['selected']
    .diff()
    .fillna(0)
)

player_df_complete = player_df_complete[player_df_complete['team_id'] != 0].copy()

# Add opponent details
# Aggregate player data so there is exactly ONE row per team per round
team_lookup = player_df_complete.groupby(['team', 'round']).agg({
    'clean_sheets': 'max',
    'elo': 'first', # Elo is usually the same for all players on the team
    'strength': 'first'
}).reset_index()

# Rename columns to 'opponent_...' so they don't clash with the player's own stats
team_lookup.columns = ['opponent_team_name', 'round'] + [f'opp_{col}' for col in team_lookup.columns if col not in ['team', 'round']]

# check your column name for who the player is playing AGAINST (e.g., 'opponent')
player_df_final = player_df_complete.merge(
    team_lookup,
    on=['opponent_team_name', 'round'],  # 'opponent' is the ID of the team they face
    # on=['opponent_team_name', 'round'],
    how='left'
)


#Fill missing values
numeric_cols = player_df_final.select_dtypes(include=['number']).columns

# # Tier 1: Player-level mean
player_df_final[numeric_cols] = (
                                player_df_final[numeric_cols]
                                .fillna(
                                    player_df_final
                                    .groupby('fpl_id')[numeric_cols]
                                    .transform(lambda x: x.fillna(x.expanding().mean()))
                    ))

# # Tier 2: Team-level mean (for players with NO personal stats yet)
player_df_final[numeric_cols] = (
                                player_df_final[numeric_cols]
                                .fillna(
                                    player_df_final
                                    .groupby(['team', 'round'])[numeric_cols]
                                    .transform(lambda x: x.fillna(x.expanding().mean()))
                    ))

# # Tier 3: Global constant (The safety net)
player_df_final[numeric_cols] = player_df_final[numeric_cols].fillna(0)

player_df_final.to_csv('./data/rolled_player_data.csv', index=False)
player_df_final

,fpl_id,round,first_name,second_name,web_name,now_cost,selected_by_percent,form,event_points,transfers_in_event,...,strength_ewm,strength_overall_home_ewm,strength_overall_away_ewm,strength_attack_home_ewm,strength_attack_away_ewm,strength_defence_home_ewm,strength_defence_away_ewm,opp_clean_sheets,opp_elo,opp_strength
0,1.0,1,David,Raya Martín,Raya,6.0,36.9,3.2,10.0,418466.0,...,0.000000,0.00,0.000,0.00,0.00,0.00,0.0,0.000000,0.000000,0.000000
1,1.0,2,David,Raya Martín,Raya,5.5,21.5,8.0,6.0,77335.0,...,4.000000,1320.00,1325.000,1350.00,1350.00,1290.00,1300.0,1.000000,2037.000000,4.000000
2,1.0,3,David,Raya Martín,Raya,5.5,21.8,6.0,2.0,38830.0,...,4.000000,1320.00,1325.000,1350.00,1350.00,1290.00,1300.0,1.000000,1993.000000,5.000000
3,1.0,4,David,Raya Martín,Raya,5.5,23.5,6.0,6.0,33143.0,...,4.000000,1320.00,1325.000,1350.00,1350.00,1290.00,1300.0,1.000000,2037.000000,4.000000
4,1.0,5,David,Raya Martín,Raya,5.5,24.0,4.0,2.0,95889.0,...,4.000000,1320.00,1325.000,1350.00,1350.00,1290.00,1300.0,0.000000,2037.000000,4.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6749,792.0,22,Alysson Edward Franco,da Rocha dos Santos,Alysson,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.00,0.000,0.00,0.00,0.00,0.0,0.503854,1876.138358,3.255263
6750,793.0,22,George,King,King,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.00,0.000,0.00,0.00,0.00,0.0,0.460100,1827.741779,3.101951
6751,794.0,21,Max,Alleyne,Alleyne,4.5,0.0,4.0,4.0,745.0,...,0.000000,0.00,0.000,0.00,0.00,0.00,0.0,0.274660,1842.056122,3.176020
6752,794.0,22,Max,Alleyne,Alleyne,0.0,0.0,4.0,0.0,0.0,...,0.727273,156.25,163.125,151.25,153.75,161.25,172.5,0.262236,1841.822055,3.141567


In [ ]:
player_df_final.columns.tolist()

# 24/25


In [2]:
# players = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2024-2025/By%20Tournament/Premier%20League/GW1/players.csv')
# teams = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2024-2025/By%20Tournament/Premier%20League/GW1/teams.csv')
master_fpl_understat_map = pd.read_csv('https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/main/Master.csv')
git_data_olbauday_base = "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/2024-2025/"

git_vaastav_base = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/refs/heads/master/data/2024-25/"


fpl_fixtures_url = 'https://fantasy.premierleague.com/api/fixtures/?event='

player_summary = 'https://fantasy.premierleague.com/api/element-summary/'

# bootstrap = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/').json()
# bootstrap

## Understat


In [4]:
# Initialize the client
with UnderstatClient() as understat:
    # Use .league() to specify the league, then .get_player_data() for the season
    data = understat.league(league="EPL").get_player_data(season="2024")

# This will return a list of dictionaries containing player stats
understat_data = pd.DataFrame(data)

# List of IDs from your data
player_ids = understat_data['id'].values #['8260', '13222', '11363'] # Haaland, Thiago, Semenyo, etc.
player_name_id = understat_data.set_index('id')['player_name']
player_name_id
all_history = []

with UnderstatClient() as understat:
    for p_id in player_ids:
        # This gets the season-by-season history for that specific ID
        history = understat.player(player=p_id).get_match_data()

        # # Add the player name back in so you know who is who
        for gw_ in history:
            gw_['understat_id'] = p_id
            gw_['understat_name'] = player_name_id[p_id]
            all_history.append(gw_)

# # Convert to a history DataFrame
history_df = pd.DataFrame(all_history)
history_df.to_csv(f'./data/understat/understat_24_25.csv', index=False)


KeyboardInterrupt: 

## FPL


In [3]:
all_player_gw_stats = []
for round_ in range(1,39):
    player_gw_stats = pd.read_csv(f'{git_vaastav_base}gws/gw{round_}.csv')
    player_gw_stats['round'] = round_
    all_player_gw_stats.append(player_gw_stats)

all_player_gw_stats = pd.concat(all_player_gw_stats, ignore_index=True)

teams = pd.read_csv(f'{git_vaastav_base}teams.csv')
teams = teams[['id', 'name', 'short_name', 'code']]
teams_id_map = pd.Series(teams['name'].values, index=teams['id']).to_dict()
teams_id_map

# all_player_gw_stats['team_name'] = all_player_gw_stats['team'].map(teams_id_map)
all_player_gw_stats['opponent_team'] = all_player_gw_stats['opponent_team'].map(teams_id_map)

# Get team_a and team_h
all_player_gw_stats['team_h'] = all_player_gw_stats.apply(lambda row: row['team'] if row['was_home'] else row['opponent_team'], axis=1)
all_player_gw_stats['team_a'] = all_player_gw_stats.apply(lambda row: row['opponent_team'] if row['was_home'] else row['team'], axis=1)
all_player_gw_stats

,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,yellow_cards,mng_clean_sheets,mng_draw,mng_goals_scored,mng_loss,mng_underdog_draw,mng_underdog_win,mng_win,team_h,team_a
0,Alex Scott,MID,Bournemouth,1.6,0,0,11,0,12.8,77,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Nott'm Forest,Bournemouth
1,Carlos Miguel dos Santos Pereira,GK,Nott'm Forest,2.2,0,0,0,0,0.0,427,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Nott'm Forest,Bournemouth
2,Tomiyasu Takehiro,DEF,Arsenal,0.0,0,0,0,0,0.0,22,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Arsenal,Wolves
3,Malcolm Ebiowei,MID,Crystal Palace,0.0,0,0,0,0,0.0,197,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Brentford,Crystal Palace
4,Ben Brereton Díaz,MID,Southampton,1.0,0,0,-2,0,14.0,584,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Newcastle,Southampton
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27600,Ashley Young,DEF,Everton,3.3,0,0,23,1,24.8,238,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Newcastle,Everton
27601,Somto Boniface,DEF,Ipswich,-0.5,0,0,0,0,0.0,799,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ipswich,West Ham
27602,Simon Rusk,AM,Southampton,1.3,0,0,0,0,0.0,748,...,0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,Southampton,Arsenal
27603,Arne Slot,AM,Liverpool,4.0,0,0,0,0,0.0,744,...,0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,Liverpool,Crystal Palace


In [27]:

teams = pd.read_csv(f'{git_vaastav_base}teams.csv')
teams = teams[['id', 'name', 'short_name', 'code']]
teams_id_map = pd.Series(teams['name'].values, index=teams['id']).to_dict()
teams_id_map

all_player_gw_stats_ = all_player_gw_stats.copy()

# Get team_a and team_h
all_player_gw_stats_['team_h'] = all_player_gw_stats_.apply(lambda row: row['team'] if row['was_home'] else row['opponent_team'], axis=1)
all_player_gw_stats_['team_a'] = all_player_gw_stats_.apply(lambda row: row['opponent_team'] if row['was_home'] else row['team'], axis=1)


fixtures = pd.read_csv(f'{git_vaastav_base}fixtures.csv')
fixtures = fixtures[['event', 'team_a', 'team_h','team_a_score', 'team_h_score', 'team_a_difficulty', 'team_h_difficulty']]
fixtures = fixtures.rename({'event': 'round'}, axis=1)

fixtures['team_a'] = fixtures['team_a'].map(teams_id_map)
fixtures['team_h'] = fixtures['team_h'].map(teams_id_map)


all_player_gw_stats_clean = all_player_gw_stats_.merge(fixtures[['team_a', 'team_h', 'round', 'team_a_difficulty', 'team_h_difficulty']],
                                                    on=['team_a', 'team_h', 'round'],
                                                    how='left')
all_player_gw_stats_clean = all_player_gw_stats_clean.rename({'element': 'fpl_id'}, axis=1)



understat_data = pd.read_csv(f'./data/understat/understat_24_25.csv')
fpl_understat_map = master_fpl_understat_map[~master_fpl_understat_map['24-25'].isna()][['24-25', 'understat', 'web_name']].rename({'24-25': 'fpl_id', 'understat':'understat_id', 'web_name': 'fpl_name'}, axis=1)

understat_data = understat_data[understat_data['season'] == 2024]
understat_data = understat_data.merge(fpl_understat_map[['fpl_id', 'understat_id']],
                                     on='understat_id',
                                      how='left')

fpl_understat_map = master_fpl_understat_map[~master_fpl_understat_map['24-25'].isna()][['24-25', 'understat', 'web_name']].rename({'24-25': 'fpl_id', 'understat':'understat_id', 'web_name': 'fpl_name'}, axis=1)
all_player_gw_stats_clean = all_player_gw_stats_clean.merge(fpl_understat_map[['fpl_id', 'understat_id', 'fpl_name']],
                                    on='fpl_id',
                                    how='left')
# Remove null understat ids
all_player_gw_stats_clean = all_player_gw_stats_clean[all_player_gw_stats_clean['understat_id'].notna()]
all_player_gw_stats_clean['date'] = pd.to_datetime(all_player_gw_stats_clean['kickoff_time']).dt.date
understat_data['date'] = pd.to_datetime(understat_data['date']).dt.date

all_player_data = all_player_gw_stats_clean.merge(understat_data[[
                                                    'understat_id', 'fpl_id', 'date','shots', 'xG', 'season',
                                                    'xA', 'key_passes', 'npg', 'npxG', 'xGChain', 'xGBuildup']
                                                    ],
                                                    on=['understat_id', 'fpl_id', 'date'],
                                                    how='left'
                                                    )
all_player_data = all_player_data[all_player_data['minutes'] > 0]
all_player_data

,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,fpl_id,...,date,shots,xG,season,xA,key_passes,npg,npxG,xGChain,xGBuildup
0,Alex Scott,MID,Bournemouth,1.6,0,0,11,0,12.8,77,...,2024-08-17,0.0,0.000000,2024.0,0.056977,1.0,0.0,0.000000,0.114473,0.057496
4,Ben Brereton Díaz,MID,Southampton,1.0,0,0,-2,0,14.0,584,...,2024-08-17,2.0,0.810859,2024.0,0.076800,1.0,0.0,0.810859,0.933370,0.045711
5,Pau Torres,DEF,Aston Villa,1.9,0,0,17,0,1.9,52,...,2024-08-17,0.0,0.000000,2024.0,0.000000,0.0,0.0,0.000000,1.071885,1.071885
8,Hwang Hee-chan,MID,Wolves,1.3,0,0,14,0,16.3,550,...,2024-08-17,0.0,0.000000,2024.0,0.248339,1.0,0.0,0.000000,0.248339,0.000000
11,João Victor Gomes da Silva,MID,Wolves,0.6,0,0,11,0,3.2,553,...,2024-08-17,0.0,0.000000,2024.0,0.000000,0.0,0.0,0.000000,0.248339,0.248339
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27005,Romain Esse,MID,Crystal Palace,1.8,0,0,3,0,1.4,730,...,2025-05-25,0.0,0.000000,2024.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000
27007,Raheem Sterling,MID,Arsenal,2.3,0,0,10,0,7.1,186,...,2025-05-25,1.0,0.087486,2024.0,0.000000,0.0,0.0,0.087486,0.286839,0.199353
27008,Raúl Jiménez,FWD,Fulham,5.0,0,0,1,0,0.6,252,...,2025-05-25,2.0,0.105204,2024.0,0.000000,0.0,0.0,0.105204,0.178922,0.073717
27010,Myles Lewis-Skelly,MID,Arsenal,3.3,0,0,5,0,1.3,597,...,2025-05-25,0.0,0.000000,2024.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000


In [24]:
player_gw_stats_list = []
for round_ in range(1,39):
    player_gw_stats = pd.read_csv(f'{git_data_olbauday_base}playermatchstats/GW{round_}/playermatchstats.csv', index_col=False)
    player_gw_stats['round'] = round_
    player_gw_stats_list.append(player_gw_stats)
player_gw_stats = pd.concat(player_gw_stats_list, ignore_index=True)
player_gw_stats

,player_id,match_id,minutes_played,goals,assists,total_shots,xg,xa,xgot,shots_on_target,...,ground_duels_won_percent,aerial_duels_won_percent,successful_dribbles_percent,tackles_won_percent,start_min,finish_min,team_goals_conceded,penalties_scored,penalties_missed,round
0,17,24-25-prem-arsenal-vs-wolverhampton-wanderers,80,1,1,5,0.35,0.37,0.50,3,...,44,0,0,100,0,80,0,0,0,1
1,4,24-25-prem-arsenal-vs-wolverhampton-wanderers,90,1,1,5,0.45,0.04,0.73,1,...,45,38,33,100,0,90,0,0,0,1
2,15,24-25-prem-arsenal-vs-wolverhampton-wanderers,90,0,0,0,0.00,0.00,0.00,0,...,0,0,0,0,0,90,0,0,0,1
3,20,24-25-prem-arsenal-vs-wolverhampton-wanderers,90,0,0,1,0.06,0.06,0.00,0,...,60,0,0,67,0,90,0,0,0,1
4,9,24-25-prem-arsenal-vs-wolverhampton-wanderers,90,0,0,1,0.08,0.19,0.00,0,...,58,67,50,50,0,90,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11562,95,24-25-prem-wolverhampton-wanderers-vs-brentford,10,0,0,0,0.00,0.06,0.00,0,...,0,0,0,0,80,91,0,0,0,38
11563,86,24-25-prem-wolverhampton-wanderers-vs-brentford,4,0,0,1,0.14,0.00,0.00,0,...,0,0,0,0,86,91,0,0,0,38
11564,556,24-25-prem-wolverhampton-wanderers-vs-brentford,1,0,0,0,0.00,0.00,0.00,0,...,0,0,0,0,91,91,0,0,0,38
11565,535,24-25-prem-wolverhampton-wanderers-vs-brentford,3,0,0,0,0.00,0.00,0.00,0,...,0,0,0,0,87,91,0,0,0,38


In [30]:
player_gw_stats_ = player_gw_stats.copy()

player_gw_stats_ = player_gw_stats_.drop_duplicates(subset=['round', 'player_id'], keep='first')
all_player_data = all_player_data.drop_duplicates(subset=['round', 'fpl_id'], keep='first')

player_gw_stats_ = player_gw_stats_.rename({'player_id': 'fpl_id'}, axis=1)

player_data = all_player_data.merge(player_gw_stats_[[
    'fpl_id', 'interceptions', 'recoveries', 'blocks', 'clearances','tackles', 'round'
]], on=['fpl_id', 'round'], how='left')

player_data.to_csv('./data/player_data_24-25.csv', index=False)

In [ ]:
player_data = pd.read_csv('./data/player_data_24-25.csv')
player_data = player_data[[
    'name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets', 'creativity', 'fpl_id', 'expected_assists',
    'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored', 'ict_index',
    'influence', 'kickoff_time', 'minutes', 'modified', 'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards',
    'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in',
    'transfers_out', 'value', 'was_home', 'yellow_cards', 'team_h', 'team_a', 'team_a_difficulty', 'team_h_difficulty', 'understat_id',
    'fpl_name', 'date', 'shots', 'xG', 'season', 'xA', 'key_passes', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'interceptions',
    'recoveries', 'blocks', 'clearances', 'tackles']].copy()
player_data_clean = player_data.dropna()
# Load the current season's data directly from the source
season = "2526" # Change this for historical seasons
url = f"https://www.football-data.co.uk/mmz4281/2425/E0.csv"

# It's good practice to use a custom User-Agent to avoid blocks
odds_data = pd.read_csv(url, low_memory=False)

# 1. Clean up dates in betting data
odds_data['Date'] = pd.to_datetime(odds_data['Date'], dayfirst=True).dt.date

# 2. Extract key columns (B365H = Bet365 Home Odds, B365D = Draw, B365A = Away)
odds_subset = odds_data[['Date', 'HomeTeam', 'AwayTeam', 'B365H', 'B365D', 'B365A']].copy()
odds_subset = odds_subset.rename(columns={'Date': 'date'})

def add_odds(row):
    win = round(1/row['B365H'], 5)
    draw = round(1/row['B365D'], 5)
    lose = round(1/row['B365A'], 5)

    # Normalize the probabilities (to make the probabilities sum to 100%)
    sum_percent = win + draw + lose
    win_prob = round(win/sum_percent, 3)
    draw_prob = round(draw/sum_percent, 3)
    lose_prob = round(lose/sum_percent, 3)

    return pd.Series([win_prob, draw_prob, lose_prob])

odds_subset[['win_prob', 'draw_prob', 'lose_prob']] = odds_subset.apply(add_odds, axis=1)


# Make sure the names match in the player_data df and odds_subset df
team_map = {
    'Arsenal': 'Arsenal',
    'Aston Villa': 'Aston Villa',
    'Bournemouth': 'Bournemouth',
    'Brighton': 'Brighton',
    'Brentford': 'Brentford',
    'Burnley': 'Burnley',
    'Chelsea': 'Chelsea',
    'Crystal Palace': 'Crystal Palace',
    'Everton': 'Everton',
    'Fulham': 'Fulham',
    'Ipswich': 'Ipswich',
    "Nott'm Forest": "Nott'm Forest",
    'Liverpool': 'Liverpool',
    'Leicester': 'Leicester',
    'Man City': 'Man City',
    'Man United': 'Man Utd',
    'Newcastle': 'Newcastle',
    'Southampton': 'Southampton',
    'Tottenham': 'Spurs',
    'West Ham': 'West Ham',
    'Wolves': 'Wolves',
}

odds_subset['HomeTeam'] = odds_subset['HomeTeam'].map(team_map)
odds_subset['AwayTeam'] = odds_subset['AwayTeam'].map(team_map)


odds_melted = odds_subset.melt(
    id_vars=['date','B365H', 'B365D', 'B365A', 'win_prob', 'draw_prob', 'lose_prob'],
    value_vars=['HomeTeam', 'AwayTeam'],
    var_name='side',
    value_name='team'
)

odds_melted['date'] = odds_melted['date'].astype(str)

player_data_odds = player_data_clean.merge(odds_melted[['date', 'win_prob', 'draw_prob', 'lose_prob','team']],
                  on=['date', 'team'],
                  how='left'
                  )

playerstats = pd.read_csv(f'{git_data_olbauday_base}playerstats/playerstats.csv')
playerstats = playerstats.rename({'id': 'fpl_id', 'gw': 'round'}, axis=1)

player_data_odds = player_data_odds.merge(playerstats[['fpl_id', 'round', 'selected_by_percent', 'form']],
                                            on=['fpl_id', 'round'],
                                            how='left')

# 1. Sort by player and round to ensure the sequence is correct
player_data_odds = player_data_odds.sort_values(['fpl_id', 'round'])

# 2. Group by player and apply the difference
player_data_odds['ownership_change'] = player_data_odds.groupby('fpl_id')['selected'].diff().fillna(0)
player_data_odds['pts_bonus'] = player_data_odds['total_points'] - player_data_odds['bonus']

def ownership_change(row):
    net_transfers = row['transfers_in'] - row['transfers_out']
    total_transfers = row['transfers_in'] + row['transfers_out']
    net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

    return net_transfers_pct

player_data_odds['percenatge_net_transfers'] = player_data_odds.apply(ownership_change, axis=1)

matches = pd.read_csv(f'{git_data_olbauday_base}matches/matches.csv')
teams = pd.read_csv(f'{git_data_olbauday_base}teams/teams.csv')

team_code_id_map = pd.Series(teams['id'].values, teams['code']).to_dict()
matches['team_h'] = matches['home_team'].map(team_code_id_map)
matches['team_a'] = matches['away_team'].map(team_code_id_map)

matches['team_h'] = matches['team_h'].map(teams_id_map)
matches['team_a'] = matches['team_a'].map(teams_id_map)

def add_elo(row):
    match_ = matches[(matches['team_a'] == row['team_a']) & (matches['team_a'] == row['team_a'])]
    return pd.Series({'elo_h': match_.iloc[0]['home_team_elo'],
                      'elo_a': match_.iloc[0]['away_team_elo']
    })

player_data_odds[['elo_h', 'elo_a']] = player_data_odds.apply(add_elo, axis=1)

player_data_odds['elo'] = player_data_odds.apply(
    lambda row: row['elo_h'] if row['was_home'] else row['elo_a'],
    axis=1
)
player_data_odds.to_csv('./data/player_data_odds_24-25.csv')

### Get CBITR & PER_90


In [189]:
combined_data = pd.read_csv('./data/player_data_odds_24-25.csv', index_col=0)

# GET cbi & adjust the xP
combined_data['clearances_blocks_interceptions'] = combined_data['clearances'] + combined_data['blocks'] + combined_data['interceptions']
combined_data['cbit'] = combined_data['clearances_blocks_interceptions'] + combined_data['tackles']

def adjust_xP(row):
    if row['position'] == 'GK':
        return row['xP']

    threshold = 10 if row['position'] == 'DEF' else 12
    cbitr = row['cbit'] if row['position'] == 'DEF' else row['cbit'] + row['recoveries']

    return row['xP'] + 2 if cbitr > threshold else row['xP']

combined_data['xP_25'] = combined_data.apply(adjust_xP, axis=1)

# Get the total CBIT actions for the season so far
## Cater for the mssing rounds before rolling
# 1. Get all unique players and all unique rounds
all_players = combined_data['fpl_id'].unique()
all_rounds = range(combined_data['round'].min(), combined_data['round'].max() + 1)

# 2. Create a MultiIndex of every player x every round
multi_idx = pd.MultiIndex.from_product([all_players, all_rounds], names=['fpl_id', 'round'])

# 3. Reindex the dataframe
# This inserts "empty" rows for missing player/round combinations
combined_data = combined_data.set_index(['fpl_id', 'round']).reindex(multi_idx).reset_index()

# Fill statistical columns with 0
# stat_columns = ['goals_scored', 'goals_conceded', 'expected_goals_conceded']
combined_data = combined_data.fillna(0)

# Sort to ensure chronological order for the rolling window
combined_data = combined_data.sort_values(['fpl_id', 'round'])

# Get the sum of CBIT actions over the all the previous rounds
combined_data.sort_values(['fpl_id', 'round'], inplace=True)

# 2. Group by player and apply cumsum to relevant metrics
metrics_to_cumulate = ['recoveries', 'minutes',  'tackles', 'clearances_blocks_interceptions']
for metric in metrics_to_cumulate:
    col_name = f'season_{metric}'
    combined_data[col_name] = combined_data.groupby('fpl_id')[metric].transform(lambda x: x.shift(1).cumsum())

numeric_cols = ['season_recoveries', 'season_minutes',  'season_tackles', 'season_clearances_blocks_interceptions']
for col in numeric_cols:
    combined_data[col] = pd.to_numeric(combined_data[col], errors='coerce').fillna(0)

# Base CBIT actions (Clearances + Blocks + Interceptions + Tackles)
combined_data['cbit_actions'] = combined_data['season_tackles'] + combined_data['season_clearances_blocks_interceptions']

# Total Defensive Contribution (for Defenders and Midfielders includes Recoveries)
combined_data['total_cbitr_actions'] = combined_data.apply(
            lambda x: x['cbit_actions'] + x['season_recoveries'] if x['position'] in ['DEF', 'MID'] else x['cbit_actions'],
            axis=1
        )

# Calculate volume per 90 for the probability model
combined_data['cbitr_volume_90'] = (combined_data['total_cbitr_actions'] / combined_data['season_minutes'].replace(0, np.nan)) * 90

def estimate_cbit_prob(row):
    """
    Uses the cumulative rate (historical feature) to predict the
    probability of hitting the threshold in the next match.
    """
    threshold = 10 if row['position'] == 'DEF' else 12

    # Require a sample of at least 3 games (270 mins) for the feature to be considered reliable
    if row['season_minutes'] < 270:
        return 0.0

    # lam is the 'historical feature' representing expected actions in 90 mins
    lam = row['cbitr_volume_90']

    # Poisson survival function: P(X >= threshold)
    return round(poisson.sf(threshold - 1, lam), 4)

combined_data['cbitr_probability'] = combined_data.apply(estimate_cbit_prob, axis=1)

combined_data.to_csv('./data/combined_data_24-25.csv')

## Roll values


In [212]:
# player_data_odds = pd.read_csv('./data/final_player_data.csv')
combined_data = pd.read_csv('./data/combined_data_24-25.csv', index_col=0)

rolling_features = [
    'creativity', 'influence', 'threat', 'minutes',  'pts_bonus', 'total_points',
    'expected_goals', 'expected_assists', 'xP', 'xP_25',
    'expected_goals_conceded', 'goals_conceded', 'goals_scored',
    'shots', 'key_passes', 'npg', 'npxG', 'shots', 'xG','xA',
    'saves', 'starts', 'yellow_cards', 'red_cards','assists', 'clean_sheets',

    'value', 'ict_index', 'selected', 'transfers_in', 'transfers_out',
    'ownership_change', 'percenatge_net_transfers',
    'xGChain', 'xGBuildup', 'expected_goal_involvements', 'form',
    'clearances_blocks_interceptions', 'tackles', 'recoveries','selected_by_percent',
    'goals_conceded','goals_scored',
]

player_df = combined_data.copy()

# Fill statistical columns with 0
# stat_columns = ['goals_scored', 'goals_conceded', 'expected_goals_conceded']
player_df_complete = player_df.fillna(0)

# Sort to ensure chronological order for the rolling window
player_df_complete = player_df_complete.sort_values(['fpl_id', 'round']).reset_index(drop=True)

# Averagae values
new_features = {}
for col in rolling_features:
    # The rolling window now spans actual calendar rounds
    grp = player_df_complete.groupby('fpl_id')[col]

    new_features[f'{col}_1'] = grp.transform(
        lambda x: x.shift(1).rolling(1, min_periods=1).sum()
    )

    new_features[f'{col}_3'] = grp.transform(
        lambda x: (x.shift(1).rolling(3, min_periods=1).sum())/3
    )

    new_features[f'{col}_5'] = grp.transform(
        lambda x: (x.shift(1).rolling(5, min_periods=1).sum())/5
    )

# Join all new columns at once (no fragmentation)
player_df_complete = pd.concat(
    [player_df_complete, pd.DataFrame(new_features)],
    axis=1
)

player_df_complete = player_df_complete[player_df_complete['team'] != '0']

# Ownership change per player
player_df_complete['ownership_change'] = (
    player_df_complete
    .groupby('fpl_id')['selected']
    .diff()
    .fillna(0)
)
# Add opponent details
# Aggregate player data so there is exactly ONE row per team per round
team_lookup = player_df_complete.groupby(['team', 'round']).agg({
    'clean_sheets': 'max',
    'elo_h': 'first', # Elo is usually the same for all players on the team
    'elo_a': 'first',
    'elo': 'first'
}).reset_index()

# Rename columns to 'opponent_...' so they don't clash with the player's own stats
team_lookup.columns = ['opponent_team_name', 'round'] + [f'opp_{col}' for col in team_lookup.columns if col not in ['team', 'round']]

# check your column name for who the player is playing AGAINST (e.g., 'opponent')
player_df_final = player_df_complete.merge(
    team_lookup,
    left_on=['opponent_team', 'round'],  # 'opponent' is the ID of the team they face
    right_on=['opponent_team_name', 'round'],
    how='left'
)


#Fill missing values
numeric_cols = player_df_final.select_dtypes(include=['number']).columns

# # Tier 1: Player-level mean
player_df_final[numeric_cols] = (
                                player_df_final[numeric_cols]
                                .fillna(
                                    player_df_final
                                    .groupby('fpl_id')[numeric_cols]
                                    .transform(lambda x: x.fillna(x.expanding().mean()))
                    ))

# # Tier 2: Team-level mean (for players with NO personal stats yet)
player_df_final[numeric_cols] = (
                                player_df_final[numeric_cols]
                                .fillna(
                                    player_df_final
                                    .groupby(['team', 'round'])[numeric_cols]
                                    .transform(lambda x: x.fillna(x.expanding().mean()))
                    ))

# # Tier 3: Global constant (The safety net)
player_df_final[numeric_cols] = player_df_final[numeric_cols].fillna(0)

player_df_final.to_csv('./data/rolled_player_data_24-25.csv', index=False)
player_df_final

,fpl_id,round,name,position,team,xP,assists,bonus,bps,clean_sheets,...,recoveries_3,recoveries_5,selected_by_percent_1,selected_by_percent_3,selected_by_percent_5,opponent_team_name,opp_clean_sheets,opp_elo_h,opp_elo_a,opp_elo
0,2,1,Gabriel Fernando de Jesus,FWD,Arsenal,3.2,0.0,0.0,1.0,0.0,...,0.000000,0.0,0.0,0.000000,0.00,Wolves,0.0,1810.63,1672.41,1672.41
1,2,4,Gabriel Fernando de Jesus,FWD,Arsenal,1.2,0.0,0.0,1.0,0.0,...,0.666667,0.4,0.0,0.733333,0.44,Spurs,0.0,1702.37,1981.59,1702.37
2,2,5,Gabriel Fernando de Jesus,FWD,Arsenal,0.2,0.0,0.0,3.0,0.0,...,0.333333,0.6,0.8,0.266667,0.60,Man City,0.0,1702.37,1981.59,1702.37
3,2,6,Gabriel Fernando de Jesus,FWD,Arsenal,2.2,0.0,0.0,4.0,0.0,...,1.333333,1.2,0.9,0.566667,0.78,Leicester,0.0,0.00,0.00,0.00
4,2,7,Gabriel Fernando de Jesus,FWD,Arsenal,2.3,0.0,0.0,9.0,0.0,...,1.333333,0.8,0.9,0.866667,0.52,Southampton,0.0,1695.64,1577.84,1577.84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11412,797,38,Jake Evans,MID,Leicester,0.3,0.0,0.0,3.0,0.0,...,0.000000,0.0,0.0,0.000000,0.00,Bournemouth,1.0,0.00,0.00,0.00
11413,798,32,Jay Robinson,FWD,Southampton,0.0,0.0,0.0,4.0,0.0,...,0.000000,0.0,0.0,0.000000,0.00,Aston Villa,1.0,1814.84,1868.99,1868.99
11414,798,35,Jay Robinson,FWD,Southampton,0.5,0.0,0.0,6.0,0.0,...,0.666667,0.4,0.0,0.000000,0.00,Leicester,1.0,1695.64,1577.84,1695.64
11415,798,37,Jay Robinson,FWD,Southampton,0.0,0.0,0.0,3.0,0.0,...,1.000000,1.0,0.0,0.000000,0.00,Everton,1.0,1695.64,1577.84,1695.64
